# 27 — M3 Gate

**Autoridad formal de cierre de M3.** Consume evidencia persistida por 20–26 y decide computablemente si Gold puede congelarse antes de Modeling.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

In [ ]:
from argos.evaluation.m3_gate import evaluate_m3_gate, persist_m3_gate

report, decision = evaluate_m3_gate(
    reports_root=paths.reports,
    gold_dir=paths.gold,
    manifest_dir=paths.root / "artifacts" / "manifests",
    docs_root=paths.docs,
    schema_version=GOLD_SPEC.schema_version,
)
report

In [ ]:
report_path, decision_path = persist_m3_gate(
    report=report,
    decision=decision,
    reports_root=paths.reports,
    decision_path=paths.docs / "decisions" / "M3_GATE.md",
)

print("M3 ✅ CLOSED" if decision.closed else "M3 ❌ OPEN")
print(f"Blocking checks: {decision.blocking_passed}/{decision.blocking_total}")
if decision.failed_checks:
    print("Failed:", ", ".join(decision.failed_checks))
print("Gate report:", report_path)
print("Decision artifact:", decision_path)

assert decision.closed, (
    "M3 remains OPEN. Resolve the failed checks before starting Modeling."
)

## Después del cierre

Si el gate imprime `M3 ✅ CLOSED`, congelar Gold, el test temporal y los manifests. El siguiente milestone comienza con `30_baseline_heuristics.ipynb` y `31_logistic_regression_baseline.ipynb`.